In [1]:
import cv2
import json
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
from metric_yyy import read_bb_yolo, bb_xywh2xyxyxyxy, bb_scale, single_image_confusion_matrix
from KVClusterV3 import KVClusterV3

In [3]:
TASK_TYPE = 0
TARGET_ACCURACY = 0.8
CLASS_INDEX = '2'

FEA_FEATURE_INDEX = 1
RP_AMOUNT = 4
CONFIDENCE_THRESHOD = 0.5

In [4]:
# Plot
plot_directory = 'dataset/video_result'
power_profile_directory = 'dataset/FPS-Power.json'

In [5]:
plot_filenames = sorted(os.listdir(plot_directory))
plot_video_names = sorted(list(set([f.split('_')[0] for f in plot_filenames])))

In [6]:
colors = [
	'#e6194B',
	'#9A6324',
	'#911eb4',
	'#3cb44b',
	'#f032e6',
	'#4363d8',
]

In [7]:
def load_json_file(file_path):
	try:
		with open(file_path, 'r') as file:
			data = json.load(file)
		return data
	
	except Exception as e:
		print(f"An error occurred while loading the JSON file: {e}")
		return None

In [8]:
def find_y_given_x(x_list, y_list, x_value, degree=3):
    """
    Fits a polynomial relationship between x_list and y_list, and finds the corresponding y for a given x_value.
    
    Parameters:
        x_list (list or array-like): The list of x values.
        y_list (list or array-like): The list of y values corresponding to x_list.
        x_value (float): The x value for which we want to find the corresponding y.
        degree (int): The degree of the polynomial to fit. Default is 2 (quadratic fit).
    
    Returns:
        float: The predicted y value corresponding to x_value.
    """
    # Ensure inputs are numpy arrays
    x_array = np.array(x_list)
    y_array = np.array(y_list)

    # Fit a polynomial model of the specified degree
    coefficients = np.polyfit(x_array, y_array, degree)
    polynomial = np.poly1d(coefficients)
    
    # Predict the y value for the given x_value
    y_value = polynomial(x_value)
    
    return y_value

## Algorithms

In [9]:
def extract_accuracy_reference_index(video_category, plot_video_name, clip_index, detect_reference_index):
	video_data_directory = f'../../RTX/profiler_yolov8_accuracy_movement/{video_category}_I1/{plot_video_name}_I1'
	clip_info = load_json_file(f'{video_data_directory}/Frame_Dup_I1_F30.json')
	clip_names = list(clip_info.keys())

	image_file_path = clip_info[clip_names[clip_index]]
	real_label_file_path = [p[0 : p.index('.')]+'.txt' for p in image_file_path]
	pred_label_file_path = [real_label_file_path[dfi] for dfi in detect_reference_index]

	tps, real_ps, pred_ps = [], [], []
	for image_idx in range(len(real_label_file_path)):
		image_path = f'{video_data_directory}/Frame_All_I1/{image_file_path[image_idx]}'
		bb_real_path = f'{video_data_directory}/Label_GT_I1/{real_label_file_path[image_idx]}'
		bb_pred_path = f'{video_data_directory}/Label_GT_I1/{pred_label_file_path[image_idx]}'
		
		src = cv2.imread(image_path)
		sphereH, sphereW, _ = map(int, src.shape)

		bb_real_raw = read_bb_yolo(bb_real_path)
		bb_real_8 = bb_xywh2xyxyxyxy(bb_real_raw)
		bb_real_8 = bb_scale(bb_real_8, sphereW, sphereH)

		bb_pred_raw = read_bb_yolo(bb_pred_path)
		bb_pred_8 = bb_xywh2xyxyxyxy(bb_pred_raw)
		bb_pred_8 = bb_scale(bb_pred_8, sphereW, sphereH)

		tp, real_p, pred_p = single_image_confusion_matrix(bb_real_8, bb_pred_8, int(CLASS_INDEX), CONFIDENCE_THRESHOD)
		tps.append(tp)
		real_ps.append(real_p)
		pred_ps.append(pred_p)

	tp_clip = np.sum(np.array(tps))
	rp_clip = np.sum(np.array(real_ps))
	pp_clip = np.sum(np.array(pred_ps))

	# Handle division by zero
	prec = 1.0
	if pp_clip != 0:
		prec = tp_clip / pp_clip

	# Handle division by zero
	reca = 1.0
	if rp_clip != 0:
		reca = tp_clip / rp_clip

	# Handle division by zero
	f1 = 1.0
	if prec != 0 and reca != 0:
		f1 = 2 / ( (1 / prec) + (1 / reca) )

	return f1

In [10]:
def extract_detect_reference_index(pixel_feature, clip_index, diff_threshold):
    clip_pixel_feature = pixel_feature[clip_index]
    detect_index = [i+1 for i in range(len(clip_pixel_feature)) if clip_pixel_feature[i] > diff_threshold]
    detect_index.insert(0, 0)

    clip_size = len(clip_pixel_feature) + 1
    detect_reference_index = []
    current_reference = -1
    for i in range(clip_size):
        if i in detect_index:
            current_reference = i
        detect_reference_index.append(current_reference)
    
    return detect_reference_index

In [11]:
video_category = 'EvaluationVideo'
plot_video_name = plot_video_names[0]
diff_thresholds = [0.25, 0.2, 0.15, 0.1, 0.05, 0.04, 0.03, 0.02, 0.015, 0.01, 0.005, 0.00]
num_cluster = 3
distance_threshold = 0.015
max_fps = 30

In [12]:
raw_feature_result = load_json_file(os.path.join(plot_directory, plot_video_name + "_Feature_Result.json"))[CLASS_INDEX]
test_length = len(raw_feature_result)

clip_feature = [rfr['feature'][str(max_fps)] for rfr in raw_feature_result]
clip_feature_single = [cf[FEA_FEATURE_INDEX] for cf in clip_feature] # Pixel Feature
clip_feature_single_filtered = [clip_feature_single[i] if i == 0 else clip_feature_single[i][0:-1] for i in range(len(clip_feature_single))] # Filter Inter-Video Feature
pixel_feature = [list(1 - np.array(cf)) for cf in clip_feature_single_filtered] # Pixel Feature

cluster = KVClusterV3(num_cluster)
fps_list = []
accuracy_list = []
re_train_index = []

# Inital Training
for inital_clip_index in range(RP_AMOUNT):
    re_train_index.append(inital_clip_index)
    for diff_threshold in diff_thresholds:
        detect_reference_index = extract_detect_reference_index(pixel_feature, inital_clip_index, diff_threshold)
        f1_at_diff_value = extract_accuracy_reference_index(video_category, plot_video_name, inital_clip_index, detect_reference_index)

        if f1_at_diff_value > TARGET_ACCURACY:
            average_feature = float(np.average(np.array(pixel_feature[inital_clip_index])))
            cluster.add([average_feature], [diff_threshold])
            fps_list.append(max_fps)
            accuracy_list.append(f1_at_diff_value)

            print(max_fps, f1_at_diff_value)

            break
cluster.cluster_lower_bound()

current_clip_index = RP_AMOUNT
while current_clip_index < test_length:
    average_feature = float(np.average(np.array(pixel_feature[current_clip_index])))
    min_distance, suggest_diff_value = cluster.tell_and_distance(average_feature)
    print(min_distance, suggest_diff_value)
    
    if min_distance > distance_threshold:
        # Re-training

        print("Re-training")
        for _ in range(RP_AMOUNT):
            re_train_index.append(current_clip_index)
            for diff_threshold in diff_thresholds:
                detect_reference_index = extract_detect_reference_index(pixel_feature, current_clip_index, diff_threshold)
                f1_at_diff_value = extract_accuracy_reference_index(video_category, plot_video_name, current_clip_index, detect_reference_index)

                if f1_at_diff_value > TARGET_ACCURACY:
                    average_feature = float(np.average(np.array(pixel_feature[current_clip_index])))
                    cluster.add([average_feature], [diff_threshold])
                    fps_list.append(max_fps)
                    accuracy_list.append(f1_at_diff_value)

                    print(max_fps, f1_at_diff_value)

                    break
            current_clip_index += 1
        cluster.cluster_lower_bound()
    
    else:
        detect_reference_index = extract_detect_reference_index(pixel_feature, current_clip_index, suggest_diff_value)
        f1_at_diff_value = extract_accuracy_reference_index(video_category, plot_video_name, current_clip_index, detect_reference_index)

        detect_fps = len(list(set(detect_reference_index)))
        fps_list.append(detect_fps)
        accuracy_list.append(f1_at_diff_value)
        current_clip_index += 1

        print(detect_fps, f1_at_diff_value)

30 1.0
30 1.0
30 0.9523809523809526
30 1.0
0.0003614258681965744 [0.]
29 1.0
0.0009097719665834697 [0.]
29 1.0
0.0018900822819682148 [0.005]
21 1.0
0.0023080925682999708 [0.01]
25 0.9459459459459459
0.03200581502411204 [0.01]
Re-training
30 0.8671328671328671
30 0.8043478260869565
30 1.0
30 0.9743589743589742
0.0025115484851458824 [0.]
29 1.0
0.009792438731096507 [0.02]
16 0.8711656441717791
0.008453923489077293 [0.]
29 1.0
0.009154194310841047 [0.]
29 1.0
0.00029604660973042707 [0.04]
18 0.660377358490566
0.000581262250753771 [0.04]
16 0.5316455696202532
0.010334532346189104 [0.]
29 1.0
0.0032063903568417505 [0.]
29 1.0
0.007405314084763311 [0.]
29 1.0
0.0055607064200062645 [0.04]
11 0.5
0.007350167556257239 [0.04]
15 0.5974025974025975
0.0008289175956923099 [0.]
29 1.0
0.006164394889103263 [0.02]
18 0.7294117647058823
0.007393186665163445 [0.02]
14 0.625
0.0061056487027409834 [0.]
29 1.0
0.001845948698062351 [0.]
29 1.0
0.005971022248379424 [0.]
29 1.0
0.0024995678347544403 [0.02]
25

In [13]:
old_power_profile = load_json_file(os.path.join(power_profile_directory))
profiler_fps_list = [int(f) for f in list(old_power_profile.keys())]
profiler_power_list = [old_power_profile[str(f)] for f in profiler_fps_list]

for i in range(profiler_fps_list[-1]+1, max_fps+1):
    iy = find_y_given_x(profiler_fps_list, profiler_power_list, i)
    profiler_fps_list.append(i)
    profiler_power_list.append(iy)

new_power_profile = {}
for i in range(len(profiler_fps_list)):
    new_power_profile[profiler_fps_list[i]] = profiler_power_list[i]

In [14]:
power_list = [new_power_profile[fps] for fps in fps_list]

In [15]:
average_accuracy = np.mean(np.array(accuracy_list))
average_power = np.mean(np.array(power_list))

print(average_accuracy, average_power)

0.9735880607427458 20.021475885219935
